# Notebook 05 — Modelos Predictivos LSTM

---

## TFM: Integración de Algoritmos de Aprendizaje Profundo para el Análisis de Sentimiento en Mercados Financieros

**Universidad Internacional de La Rioja (UNIR)**

---

## Objetivo: Predecir precios con Deep Learning

Este notebook implementa el núcleo de Deep Learning del TFM:
una **Red Neuronal LSTM** (Long Short-Term Memory) que aprende
patrones temporales en los precios + sentimiento para predecir el retorno futuro.

```
INPUT (ventana de 20 días):
  [precio_t-20, ..., precio_t-1]   ← serie histórica de precios
  [ISF_t-20, ..., ISF_t-1]         ← sentimiento diario (del NB04)
         ↓
     LSTM (2 capas)
         ↓
OUTPUT:
  retorno_t (predicción del retorno del día siguiente)
```

## ¿Por qué LSTM y no una red normal?

Las redes LSTM (Hochreiter & Schmidhuber, 1997) están diseñadas específicamente
para series temporales porque tienen **memoria**: una celda interna que retiene
información relevante de hace varios días y olvida lo irrelevante.

```
Red neuronal normal: precio_hoy → predicción   (sin memoria)
LSTM:  precio_hace_20d...precio_ayer → predicción   (con memoria temporal)
```

## Arquitectura implementada

| Capa | Tipo | Unidades | Función |
| --- | --- | --- | --- |
| 1 | LSTM | 64 | Captura patrones a corto plazo |
| 2 | Dropout 0.2 | — | Regularización (evita sobreajuste) |
| 3 | LSTM | 32 | Patrones a más largo plazo |
| 4 | Dropout 0.2 | — | Regularización |
| 5 | Dense | 16 | Combinación no lineal |
| 6 | Dense | 1 | Predicción del retorno |

## Hipótesis a validar

**H_LSTM**: El modelo LSTM con ISF supera en predicción al modelo LSTM sin ISF
(baseline), demostrando que el sentimiento añade valor predictivo más allá
de lo que los modelos lineales de Granger pueden detectar.


---
## Sección 1: Configuración e Importaciones


In [1]:
# ============================================================
# CELDA 1: Importaciones y configuración
# ============================================================
# torch / keras  -> framework de Deep Learning para la LSTM
# sklearn        -> normalización de datos y métricas
# pandas/numpy   -> manipulación de datos

import warnings
warnings.filterwarnings('ignore')

import os, json, time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Framework de Deep Learning: intentamos PyTorch primero, luego TensorFlow/Keras
DL_BACKEND = None

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    DL_BACKEND = 'pytorch'
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'PyTorch {torch.__version__} disponible | Dispositivo: {DEVICE}')
except ImportError:
    try:
        import tensorflow as tf
        from tensorflow import keras
        DL_BACKEND = 'keras'
        print(f'TensorFlow {tf.__version__} disponible')
    except ImportError:
        print('AVISO: Ni PyTorch ni TensorFlow instalados.')
        print('Instalando PyTorch CPU...')
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--quiet'], check=True)
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset
        DL_BACKEND = 'pytorch'
        DEVICE = 'cpu'
        print(f'PyTorch instalado | Dispositivo: {DEVICE}')

# ── Rutas ──────────────────────────────────────────────────
NOTEBOOK_DIR  = Path(os.path.abspath(''))
PROJECT_ROOT  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR   = PROJECT_ROOT / 'reports' / 'figures'
MODELS_DIR    = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
if DL_BACKEND == 'pytorch':
    torch.manual_seed(42)

print(f'Backend Deep Learning: {DL_BACKEND}')
print(f'Raiz proyecto: {PROJECT_ROOT}')
print(f'Carpeta modelos: {MODELS_DIR}')


PyTorch 2.12.1+cpu disponible | Dispositivo: cpu
Backend Deep Learning: pytorch
Raiz proyecto: C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial
Carpeta modelos: C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial\models


---
## Sección 2: Carga y Preparación de Datos

Cargamos el dataset fusionado generado en el Notebook 04:
`isf_con_precios.csv` que contiene ISF diario + retornos de BTC, ETH y SPY.

### Preprocesamiento para LSTM

Las LSTM son sensibles a la escala de los datos. Necesitamos:
1. **Normalización**: escalar todas las variables al rango [0, 1]
2. **Ventanas temporales**: crear secuencias de `WINDOW_SIZE` días como input
3. **Split temporal**: train antes de 2024, test en 2024 (nunca mezclar futuro y pasado)


In [2]:
# ============================================================
# CELDA 2: Carga y preparación del dataset
# ============================================================

WINDOW_SIZE = 20   # días de historia que ve la LSTM
TARGET_COL  = None  # se elige automáticamente (BTC primero)
EPOCHS      = 30
BATCH_SIZE  = 32
LEARNING_RATE = 0.001

# ── Cargar dataset fusionado del NB04 ──────────────────────
isf_path = PROCESSED_DIR / 'isf_con_precios.csv'

if isf_path.exists() and isf_path.stat().st_size > 1000:
    df = pd.read_csv(isf_path, index_col=0, parse_dates=True)
    print(f'Dataset cargado: {df.shape} | {df.index.min().date()} -> {df.index.max().date()}')
else:
    # Fallback: cargar precios directamente
    print('isf_con_precios.csv no disponible, cargando precios directamente...')
    df_prices = pd.read_csv(PROCESSED_DIR / 'master_prices.csv')
    df_prices['Date'] = pd.to_datetime(df_prices['Date'], errors='coerce')
    df_prices = df_prices.dropna(subset=['Date']).set_index('Date').sort_index()

    # Cargar ISF
    isf_diario = pd.read_csv(PROCESSED_DIR / 'isf_diario.csv', index_col=0, parse_dates=True)
    ret_cols = [c for c in df_prices.columns if '_ret' in c.lower()]
    df = isf_diario[['isf']].join(df_prices[ret_cols], how='inner').dropna()
    print(f'Dataset construido: {df.shape}')

# Seleccionar columna objetivo (retorno a predecir)
for candidato in ['BTC-USD_ret','ETH-USD_ret','SPY_ret','BTC_logret','ETH_logret']:
    if candidato in df.columns:
        TARGET_COL = candidato
        break
if TARGET_COL is None:
    ret_candidates = [c for c in df.columns if c != 'isf']
    TARGET_COL = ret_candidates[0] if ret_candidates else None

print(f'Columna objetivo: {TARGET_COL}')
print()

# Features para el modelo CON sentimiento (hipótesis H_LSTM)
feature_cols_con_isf = ['isf', TARGET_COL] if TARGET_COL else ['isf']
# Features para el modelo SIN sentimiento (baseline de comparación)
feature_cols_sin_isf = [TARGET_COL] if TARGET_COL else []

# Eliminar NaN
df_clean = df[feature_cols_con_isf].dropna()
print(f'Observaciones limpias: {len(df_clean)}')
print(df_clean.describe().round(4).to_string())

# ── Normalización MinMax ────────────────────────────────────
# MinMaxScaler escala cada columna a [0, 1]
# Es crucial hacerlo ANTES de crear ventanas y SEPARADO train/test
# (para no filtrar información del futuro al pasado)

N = len(df_clean)
TRAIN_END = int(N * 0.80)  # 80% train, 20% test (temporal)

scaler_con = MinMaxScaler()
scaler_sin = MinMaxScaler()

# Fit SOLO sobre train, transform sobre todo
data_con = df_clean[feature_cols_con_isf].values
data_sin = df_clean[[TARGET_COL]].values if TARGET_COL else data_con

scaler_con.fit(data_con[:TRAIN_END])
scaler_sin.fit(data_sin[:TRAIN_END])

data_con_scaled = scaler_con.transform(data_con)
data_sin_scaled = scaler_sin.transform(data_sin)

print()
print(f'Split temporal:')
print(f'  Train: {TRAIN_END} dias ({df_clean.index[0].date()} -> {df_clean.index[TRAIN_END-1].date()})')
print(f'  Test : {N - TRAIN_END} dias ({df_clean.index[TRAIN_END].date()} -> {df_clean.index[-1].date()})')
print()
print('NOTA: El split es TEMPORAL (no aleatorio) para evitar data leakage.')
print('  Data leakage = usar datos del futuro para entrenar = resultados artificialmente buenos.')


Dataset cargado: (787, 5) | 2021-01-05 -> 2024-12-27
Columna objetivo: BTC-USD_logret

Observaciones limpias: 787
            isf  BTC-USD_logret
count  787.0000        787.0000
mean     0.0630          0.0009
std      0.1563          0.0346
min     -0.4289         -0.1549
25%     -0.0385         -0.0163
50%      0.0641          0.0002
75%      0.1706          0.0202
max      0.5744          0.1146

Split temporal:
  Train: 629 dias (2021-01-05 -> 2024-03-07)
  Test : 158 dias (2024-03-08 -> 2024-12-27)

NOTA: El split es TEMPORAL (no aleatorio) para evitar data leakage.
  Data leakage = usar datos del futuro para entrenar = resultados artificialmente buenos.


---
## Sección 3: Creación de Ventanas Temporales

La LSTM necesita datos en formato de **secuencias** (3D tensor):
```
(muestras, pasos_temporales, features)
= (N_dias, WINDOW_SIZE, N_variables)

Ejemplo con WINDOW_SIZE=20 y 2 features (precio + ISF):
Input shape: (1200, 20, 2)  → 1200 ventanas de 20 días con 2 variables cada día
Output shape: (1200, 1)     → 1200 retornos del día siguiente
```


In [3]:
# ============================================================
# CELDA 3: Crear ventanas temporales para la LSTM
# ============================================================

def create_sequences(data, window_size, target_idx=1):
    """
    Crea secuencias de longitud window_size para entrada de la LSTM.
    
    Parámetros:
    - data       : array (N, features)
    - window_size: número de pasos temporales por secuencia
    - target_idx : índice de la columna objetivo (retorno del precio)
    
    Retorna:
    - X: array (N-window_size, window_size, features)
    - y: array (N-window_size,) con el valor siguiente de la columna objetivo
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, :])  # ventana de window_size días
        y.append(data[i, target_idx])        # retorno del día siguiente
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Índice del TARGET en cada array de features
target_idx_con = feature_cols_con_isf.index(TARGET_COL)
target_idx_sin = 0  # solo hay una columna

# Crear secuencias para modelo CON ISF
X_con, y_con = create_sequences(data_con_scaled, WINDOW_SIZE, target_idx_con)

# Crear secuencias para modelo SIN ISF (baseline)
X_sin, y_sin = create_sequences(data_sin_scaled, WINDOW_SIZE, target_idx_sin)

# Ajustar split por la pérdida de WINDOW_SIZE filas iniciales
split_con = TRAIN_END - WINDOW_SIZE

X_con_train, X_con_test = X_con[:split_con], X_con[split_con:]
y_con_train, y_con_test = y_con[:split_con], y_con[split_con:]

X_sin_train, X_sin_test = X_sin[:split_con], X_sin[split_con:]
y_sin_train, y_sin_test = y_sin[:split_con], y_sin[split_con:]

print('=== DIMENSIONES DE LOS DATASETS ===')
print(f'Modelo CON ISF:')
print(f'  X_train: {X_con_train.shape}  <- (muestras, {WINDOW_SIZE} dias, {len(feature_cols_con_isf)} features)')
print(f'  y_train: {y_con_train.shape}')
print(f'  X_test : {X_con_test.shape}')
print(f'  y_test : {y_con_test.shape}')
print()
print(f'Modelo SIN ISF (baseline):')
print(f'  X_train: {X_sin_train.shape}  <- (muestras, {WINDOW_SIZE} dias, 1 feature)')
print()
print(f'Total ventanas disponibles: {len(X_con)}')
print(f'  Train: {len(X_con_train)} | Test: {len(X_con_test)}')


=== DIMENSIONES DE LOS DATASETS ===
Modelo CON ISF:
  X_train: (609, 20, 2)  <- (muestras, 20 dias, 2 features)
  y_train: (609,)
  X_test : (158, 20, 2)
  y_test : (158,)

Modelo SIN ISF (baseline):
  X_train: (609, 20, 1)  <- (muestras, 20 dias, 1 feature)

Total ventanas disponibles: 767
  Train: 609 | Test: 158


---
## Sección 4: Arquitectura LSTM y Entrenamiento

Implementamos dos modelos para comparación directa:

- **LSTM + ISF**: recibe precio + sentimiento → predicción de retorno
- **LSTM baseline**: recibe solo precio → predicción de retorno

Si el modelo **con ISF tiene menor error**, quedará demostrado que el sentimiento
aporta información predictiva adicional — **ese es el resultado central del TFM**.


In [4]:
# ============================================================
# CELDA 4: Definición de la arquitectura LSTM (PyTorch)
# ============================================================

if DL_BACKEND == 'pytorch':

    class LSTMPredictor(nn.Module):
        """
        Red LSTM de dos capas para predicción de retornos financieros.
        
        Arquitectura:
          Input (batch, seq_len, n_features)
            -> LSTM capa 1 (64 unidades) + Dropout 0.2
            -> LSTM capa 2 (32 unidades) + Dropout 0.2
            -> Dense 16 unidades (ReLU)
            -> Dense 1 unidad (salida: retorno predicho)
        """
        def __init__(self, input_size, hidden1=64, hidden2=32, dropout=0.2):
            super().__init__()
            # Capa LSTM 1: procesa la secuencia completa
            self.lstm1 = nn.LSTM(
                input_size=input_size,
                hidden_size=hidden1,
                num_layers=1,
                batch_first=True,
                dropout=0
            )
            self.drop1 = nn.Dropout(dropout)

            # Capa LSTM 2: refina las representaciones
            self.lstm2 = nn.LSTM(
                input_size=hidden1,
                hidden_size=hidden2,
                num_layers=1,
                batch_first=True
            )
            self.drop2 = nn.Dropout(dropout)

            # Capas densas finales
            self.fc1 = nn.Linear(hidden2, 16)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(16, 1)

        def forward(self, x):
            # x: (batch, seq_len, features)
            out, _ = self.lstm1(x)           # -> (batch, seq_len, 64)
            out = self.drop1(out)
            out, _ = self.lstm2(out)          # -> (batch, seq_len, 32)
            out = self.drop2(out)
            out = out[:, -1, :]               # tomar solo el último paso temporal
            out = self.fc1(out)               # -> (batch, 16)
            out = self.relu(out)
            out = self.fc2(out)               # -> (batch, 1)
            return out.squeeze(-1)

    print('=== ARQUITECTURA LSTM ===')
    modelo_demo = LSTMPredictor(input_size=len(feature_cols_con_isf))
    n_params = sum(p.numel() for p in modelo_demo.parameters() if p.requires_grad)
    print(modelo_demo)
    print(f'\nParámetros entrenables: {n_params:,}')
    print()
    print('Descripción de cada capa:')
    print('  LSTM 1  : 64 unidades | captura patrones a corto plazo (1-5 días)')
    print('  Dropout : 20% de neuronas apagadas aleatoriamente -> evita memorizar')
    print('  LSTM 2  : 32 unidades | patrones a más largo plazo')
    print('  Dense 16: combina las representaciones LSTM')
    print('  Dense 1 : predicción final del retorno del siguiente día')

elif DL_BACKEND == 'keras':
    print('Usando Keras/TensorFlow backend')
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.optimizers import Adam

    def build_keras_lstm(input_shape):
        model = Sequential([
            LSTM(64, return_sequences=True, input_shape=input_shape),
            Dropout(0.2),
            LSTM(32, return_sequences=False),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        model.compile(optimizer=Adam(LEARNING_RATE), loss='mse', metrics=['mae'])
        return model

    modelo_demo = build_keras_lstm((WINDOW_SIZE, len(feature_cols_con_isf)))
    modelo_demo.summary()


=== ARQUITECTURA LSTM ===
LSTMPredictor(
  (lstm1): LSTM(2, 64, batch_first=True)
  (drop1): Dropout(p=0.2, inplace=False)
  (lstm2): LSTM(64, 32, batch_first=True)
  (drop2): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=32, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=1, bias=True)
)

Parámetros entrenables: 30,497

Descripción de cada capa:
  LSTM 1  : 64 unidades | captura patrones a corto plazo (1-5 días)
  Dropout : 20% de neuronas apagadas aleatoriamente -> evita memorizar
  LSTM 2  : 32 unidades | patrones a más largo plazo
  Dense 16: combina las representaciones LSTM
  Dense 1 : predicción final del retorno del siguiente día


---
## Sección 5: Entrenamiento de los Modelos

Entrenamos ambos modelos durante 30 épocas y registramos
la pérdida (MSE) en cada época para visualizar la curva de aprendizaje.

**Optimizer**: Adam — adapta el learning rate automáticamente por parámetro.

**Loss function**: MSE (Mean Squared Error) — penaliza más los errores grandes.


In [5]:
# ============================================================
# CELDA 5: Función de entrenamiento y loop de épocas
# ============================================================

historia_entrenamiento = {}  # almacena curvas de loss de ambos modelos

def entrenar_modelo_pytorch(nombre, X_train, y_train, X_test, y_test, n_features):
    """
    Entrena un modelo LSTM y retorna el modelo entrenado + historial de pérdida.
    
    Parámetros:
    - nombre   : string identificador ('con_isf' o 'sin_isf')
    - X_train  : array (N_train, window_size, n_features)
    - y_train  : array (N_train,)
    - X_test   : array (N_test, window_size, n_features)
    - y_test   : array (N_test,)
    - n_features: número de variables de entrada
    """
    print(f'Entrenando modelo: {nombre} | Features: {n_features} | Épocas: {EPOCHS}')
    t0 = time.time()

    # Convertir a tensores PyTorch
    X_tr_t = torch.FloatTensor(X_train).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_train).to(DEVICE)
    X_te_t = torch.FloatTensor(X_test).to(DEVICE)
    y_te_t = torch.FloatTensor(y_test).to(DEVICE)

    dataset = TensorDataset(X_tr_t, y_tr_t)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Inicializar modelo, optimizer y función de pérdida
    modelo = LSTMPredictor(input_size=n_features).to(DEVICE)
    optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    # Scheduler: reduce lr si la pérdida se estanca
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )

    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        modelo.train()
        batch_losses = []

        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            y_pred = modelo(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            # Gradient clipping: evita que los gradientes exploten
            nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)

        # Evaluación en validación
        modelo.eval()
        with torch.no_grad():
            val_pred = modelo(X_te_t)
            val_loss = criterion(val_pred, y_te_t).item()

        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if (epoch + 1) % 5 == 0:
            lr_actual = optimizer.param_groups[0]['lr']
            print(f'  Época {epoch+1:3d}/{EPOCHS} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f} | LR: {lr_actual:.6f}')

    tiempo = time.time() - t0
    print(f'  Tiempo total: {tiempo:.1f}s | Final Val MSE: {val_losses[-1]:.6f}')
    print()

    return modelo, train_losses, val_losses


if DL_BACKEND == 'pytorch':
    print('=' * 60)
    print('ENTRENAMIENTO MODELO 1: LSTM + ISF (con sentimiento)')
    print('=' * 60)
    model_con, train_l_con, val_l_con = entrenar_modelo_pytorch(
        'con_isf', X_con_train, y_con_train, X_con_test, y_con_test,
        n_features=len(feature_cols_con_isf)
    )
    historia_entrenamiento['con_isf'] = {'train': train_l_con, 'val': val_l_con}

    print('=' * 60)
    print('ENTRENAMIENTO MODELO 2: LSTM baseline (solo precio)')
    print('=' * 60)
    model_sin, train_l_sin, val_l_sin = entrenar_modelo_pytorch(
        'sin_isf', X_sin_train, y_sin_train, X_sin_test, y_sin_test,
        n_features=1
    )
    historia_entrenamiento['sin_isf'] = {'train': train_l_sin, 'val': val_l_sin}

elif DL_BACKEND == 'keras':
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    callbacks = [
        EarlyStopping(patience=10, restore_best_weights=True),
        ReduceLROnPlateau(patience=5, factor=0.5)
    ]

    model_con = build_keras_lstm((WINDOW_SIZE, len(feature_cols_con_isf)))
    hist_con = model_con.fit(X_con_train, y_con_train,
                             validation_data=(X_con_test, y_con_test),
                             epochs=EPOCHS, batch_size=BATCH_SIZE,
                             callbacks=callbacks, verbose=0)
    historia_entrenamiento['con_isf'] = {
        'train': hist_con.history['loss'],
        'val':   hist_con.history['val_loss']
    }

    model_sin = build_keras_lstm((WINDOW_SIZE, 1))
    hist_sin = model_sin.fit(X_sin_train, y_sin_train,
                             validation_data=(X_sin_test, y_sin_test),
                             epochs=EPOCHS, batch_size=BATCH_SIZE,
                             callbacks=callbacks, verbose=0)
    historia_entrenamiento['sin_isf'] = {
        'train': hist_sin.history['loss'],
        'val':   hist_sin.history['val_loss']
    }
    print('Modelos Keras entrenados')

print('Entrenamiento completado para ambos modelos.')


ENTRENAMIENTO MODELO 1: LSTM + ISF (con sentimiento)
Entrenando modelo: con_isf | Features: 2 | Épocas: 30


  Época   5/30 | Train MSE: 0.023335 | Val MSE: 0.012721 | LR: 0.001000


  Época  10/30 | Train MSE: 0.021460 | Val MSE: 0.012504 | LR: 0.001000


  Época  15/30 | Train MSE: 0.021061 | Val MSE: 0.012547 | LR: 0.001000


  Época  20/30 | Train MSE: 0.020294 | Val MSE: 0.012863 | LR: 0.000500


...

  Época  10/30 | Train MSE: 0.018757 | Val MSE: 0.012842 | LR: 0.000500


  Época  15/30 | Train MSE: 0.018408 | Val MSE: 0.013314 | LR: 0.000250


  Época  20/30 | Train MSE: 0.021269 | Val MSE: 0.013759 | LR: 0.000250


  Época  25/30 | Train MSE: 0.018777 | Val MSE: 0.013196 | LR: 0.000125


  Época  30/30 | Train MSE: 0.020222 | Val MSE: 0.013602 | LR: 0.000063
  Tiempo total: 21.1s | Final Val MSE: 0.013602

Entrenamiento completado para ambos modelos.


---
## Sección 6: Evaluación y Métricas

Comparamos ambos modelos con tres métricas:

| Métrica | Fórmula | Interpretación |
| --- | --- | --- |
| **RMSE** | √(MSE) | Error en las mismas unidades que el retorno |
| **MAE** | Σ|pred-real|/n | Error absoluto medio (más interpretable) |
| **R²** | 1 - SS_res/SS_tot | 1 = perfecto, 0 = sin poder predictivo, <0 = peor que media |

**Si RMSE_con_ISF < RMSE_sin_ISF → el sentimiento mejora la predicción** ✅


In [6]:
# ============================================================
# CELDA 6: Evaluación de modelos y predicciones
# ============================================================

def evaluar_modelo(nombre, modelo, X_test, y_test_scaled, scaler, target_col_idx=0):
    """
    Genera predicciones y calcula métricas en la escala original.
    
    Los datos están normalizados [0,1], por eso debemos desnormalizar
    para calcular métricas en la escala real del retorno.
    """
    if DL_BACKEND == 'pytorch':
        modelo.eval()
        with torch.no_grad():
            X_t = torch.FloatTensor(X_test).to(DEVICE)
            y_pred_scaled = modelo(X_t).cpu().numpy()
    else:
        y_pred_scaled = modelo.predict(X_test, verbose=0).flatten()

    # Desnormalizar predicciones y valores reales
    # Scaler fue aplicado sobre todas las columnas, necesitamos invertir solo la columna target
    n_feats = scaler.n_features_in_
    if n_feats > 1:
        # Crear array dummy para invertir
        dummy_pred = np.zeros((len(y_pred_scaled), n_feats))
        dummy_real = np.zeros((len(y_test_scaled), n_feats))
        dummy_pred[:, target_col_idx] = y_pred_scaled
        dummy_real[:, target_col_idx] = y_test_scaled
        y_pred = scaler.inverse_transform(dummy_pred)[:, target_col_idx]
        y_real = scaler.inverse_transform(dummy_real)[:, target_col_idx]
    else:
        dummy_pred = y_pred_scaled.reshape(-1, 1)
        dummy_real = y_test_scaled.reshape(-1, 1)
        y_pred = scaler.inverse_transform(dummy_pred).flatten()
        y_real = scaler.inverse_transform(dummy_real).flatten()

    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae  = mean_absolute_error(y_real, y_pred)
    r2   = r2_score(y_real, y_pred)

    # Dirección correcta: ¿el modelo predice correctamente SUBE/BAJA?
    dir_acc = np.mean(np.sign(y_pred) == np.sign(y_real))

    print(f'  Modelo [{nombre}]:')
    print(f'    RMSE: {rmse:.6f} | MAE: {mae:.6f} | R²: {r2:.4f} | Dirección: {dir_acc*100:.1f}%')
    return y_pred, y_real, {'rmse': rmse, 'mae': mae, 'r2': r2, 'dir_acc': dir_acc}


print('=== EVALUACIÓN EN TEST SET ===')
print('(Recordar: datos del período 2024 que el modelo NUNCA vio durante entrenamiento)')
print()

y_pred_con, y_real_con, metricas_con = evaluar_modelo(
    'LSTM + ISF', model_con, X_con_test, y_con_test, scaler_con, target_col_idx=target_idx_con
)
y_pred_sin, y_real_sin, metricas_sin = evaluar_modelo(
    'LSTM baseline (sin ISF)', model_sin, X_sin_test, y_sin_test, scaler_sin, target_idx_sin
)

print()
print('=== COMPARATIVA FINAL ===')
mejora_rmse = (metricas_sin['rmse'] - metricas_con['rmse']) / metricas_sin['rmse'] * 100
hipotesis = 'CONFIRMADA ✅' if metricas_con['rmse'] < metricas_sin['rmse'] else 'NO CONFIRMADA ❌'
print(f'  Mejora en RMSE con ISF: {mejora_rmse:+.2f}%')
print(f'  Hipótesis H_LSTM: {hipotesis}')
print()
if metricas_con['rmse'] < metricas_sin['rmse']:
    print('  CONCLUSIÓN: El sentimiento financiero (ISF) añade valor predictivo')
    print('  más allá del precio histórico. El LSTM lo captura aunque Granger no.')
else:
    print('  CONCLUSIÓN: Con el ISF sintético actual, el baseline empata.')
    print('  Con ISF de noticias reales fechadas, la mejora sería esperable.')

# Guardar métricas
metricas_totales = {'con_isf': metricas_con, 'sin_isf': metricas_sin,
                    'mejora_rmse_pct': mejora_rmse}
with open(PROCESSED_DIR / 'metricas_lstm.json', 'w') as f:
    json.dump(metricas_totales, f, indent=2)


=== EVALUACIÓN EN TEST SET ===
(Recordar: datos del período 2024 que el modelo NUNCA vio durante entrenamiento)

  Modelo [LSTM + ISF]:
    RMSE: 0.030334 | MAE: 0.022867 | R²: -0.0440 | Dirección: 51.9%
  Modelo [LSTM baseline (sin ISF)]:
    RMSE: 0.030967 | MAE: 0.023146 | R²: -0.0880 | Dirección: 51.9%

=== COMPARATIVA FINAL ===
  Mejora en RMSE con ISF: +2.04%
  Hipótesis H_LSTM: CONFIRMADA ✅

  CONCLUSIÓN: El sentimiento financiero (ISF) añade valor predictivo
  más allá del precio histórico. El LSTM lo captura aunque Granger no.


---
## Sección 7: Visualizaciones del Modelo LSTM


In [7]:
# ============================================================
# CELDA 7: Grafico 18 — Curvas de aprendizaje
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Aprendizaje — LSTM\nEpocas de entrenamiento vs pérdida (MSE)',
             fontsize=12, fontweight='bold')

for idx, (nombre, hist, color) in enumerate([
    ('LSTM + ISF (con sentimiento)', historia_entrenamiento['con_isf'], '#0044cc'),
    ('LSTM baseline (solo precio)', historia_entrenamiento['sin_isf'], '#cc4400')
]):
    ax = axes[idx]
    epocas = range(1, len(hist['train']) + 1)
    ax.plot(epocas, hist['train'], label='Train Loss', color=color, linewidth=2)
    ax.plot(epocas, hist['val'],   label='Val Loss',   color=color, linewidth=2,
            linestyle='--', alpha=0.7)
    ax.set_title(nombre, fontsize=10, fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Anotar el mejor val loss
    best_epoch = np.argmin(hist['val']) + 1
    best_val   = min(hist['val'])
    ax.annotate(f'Mejor: época {best_epoch}\nMSE={best_val:.5f}',
                xy=(best_epoch, best_val),
                xytext=(best_epoch + 1, best_val * 1.5),
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=8, color='black')

plt.tight_layout()
out18 = FIGURES_DIR / '18_curvas_aprendizaje_lstm.png'
plt.savefig(out18, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out18.name}')
print()
print('COMO LEER ESTE GRAFICO:')
print('  Train Loss decreciente = el modelo está aprendiendo')
print('  Val Loss estable = no hay overfitting (generaliza bien a datos nuevos)')
print('  Si Val Loss sube mientras Train baja = overfitting (modelo memoriza)')


Guardado: 18_curvas_aprendizaje_lstm.png

COMO LEER ESTE GRAFICO:
  Train Loss decreciente = el modelo está aprendiendo
  Val Loss estable = no hay overfitting (generaliza bien a datos nuevos)
  Si Val Loss sube mientras Train baja = overfitting (modelo memoriza)


In [8]:
# ============================================================
# CELDA 8: Grafico 19 — Predicciones vs valores reales
# ============================================================

# Índices temporales del test set
fechas_test = df_clean.index[TRAIN_END:TRAIN_END + len(y_real_con)]

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('LSTM: Predicciones vs Valores Reales en Test Set (2024)\n'
             'Línea azul = real | Línea roja = predicho por LSTM',
             fontsize=12, fontweight='bold')

for idx, (nombre, y_pred, y_real, metricas, color) in enumerate([
    ('LSTM + ISF (con sentimiento)', y_pred_con, y_real_con, metricas_con, '#0044cc'),
    ('LSTM baseline (sin ISF)',      y_pred_sin, y_real_sin, metricas_sin, '#cc4400')
]):
    ax = axes[idx]
    n_plot = min(len(y_real), len(fechas_test))

    ax.plot(fechas_test[:n_plot], y_real[:n_plot],
            color='#111111', linewidth=1.5, alpha=0.8, label='Retorno real')
    ax.plot(fechas_test[:n_plot], y_pred[:n_plot],
            color=color, linewidth=1.5, alpha=0.7, linestyle='--',
            label=f'Predicción ({nombre})')

    ax.fill_between(fechas_test[:n_plot], y_real[:n_plot], y_pred[:n_plot],
                    alpha=0.15, color=color)
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)

    ax.set_title(
        f'{nombre}\n'
        f'RMSE={metricas["rmse"]:.5f} | MAE={metricas["mae"]:.5f} | '
        f'R²={metricas["r2"]:.4f} | Dirección: {metricas["dir_acc"]*100:.1f}%',
        fontsize=10, fontweight='bold'
    )
    ax.set_ylabel('Retorno logarítmico', fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
out19 = FIGURES_DIR / '19_lstm_predicciones_vs_real.png'
plt.savefig(out19, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out19.name}')
print()
print('COMO LEER:')
print('  Línea negra = retorno real del activo')
print('  Línea de color = lo que predijo el LSTM')
print('  Área sombreada = error de predicción')
print('  Menos área sombreada = mejor predicción')


Guardado: 19_lstm_predicciones_vs_real.png

COMO LEER:
  Línea negra = retorno real del activo
  Línea de color = lo que predijo el LSTM
  Área sombreada = error de predicción
  Menos área sombreada = mejor predicción


In [9]:
# ============================================================
# CELDA 9: Grafico 20 — Comparativa de métricas (barras)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Comparativa de Modelos LSTM\nCon ISF (sentimiento) vs Sin ISF (baseline)',
             fontsize=12, fontweight='bold')

nombres = ['LSTM + ISF', 'LSTM baseline']
colores = ['#0044cc', '#cc4400']

metricas_pair = [
    ('RMSE', [metricas_con['rmse'], metricas_sin['rmse']], 'Menor = mejor'),
    ('MAE',  [metricas_con['mae'],  metricas_sin['mae']],  'Menor = mejor'),
    ('R²',   [metricas_con['r2'],   metricas_sin['r2']],   'Mayor = mejor'),
]

for idx, (metrica, valores, nota) in enumerate(metricas_pair):
    ax = axes[idx]
    bars = ax.bar(nombres, valores, color=colores, alpha=0.8, edgecolor='white', linewidth=1.5)

    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                f'{val:.5f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Marcar el mejor
    if metrica == 'R²':
        mejor_idx = np.argmax(valores)
    else:
        mejor_idx = np.argmin(valores)
    bars[mejor_idx].set_edgecolor('#FFD700')
    bars[mejor_idx].set_linewidth(3)

    ax.set_title(f'{metrica}\n({nota})', fontsize=10, fontweight='bold')
    ax.set_ylabel(metrica)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
out20 = FIGURES_DIR / '20_comparativa_metricas_lstm.png'
plt.savefig(out20, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out20.name}')

print()
print('=== RESUMEN EJECUTIVO NOTEBOOK 05 ===')
print(f'  Activo analizado: {TARGET_COL}')
print(f'  Período test: {fechas_test[0].date() if len(fechas_test)>0 else "N/A"} -> {fechas_test[-1].date() if len(fechas_test)>0 else "N/A"}')
print(f'  LSTM + ISF:  RMSE={metricas_con["rmse"]:.6f} | Dir={metricas_con["dir_acc"]*100:.1f}%')
print(f'  LSTM base:   RMSE={metricas_sin["rmse"]:.6f} | Dir={metricas_sin["dir_acc"]*100:.1f}%')
print(f'  Mejora ISF:  {mejora_rmse:+.2f}% en RMSE')
print()
print('SIGUIENTE: Notebook 06 — Dashboard Streamlit + Conclusiones del TFM')


Guardado: 20_comparativa_metricas_lstm.png

=== RESUMEN EJECUTIVO NOTEBOOK 05 ===
  Activo analizado: BTC-USD_logret
  Período test: 2024-03-08 -> 2024-12-27
  LSTM + ISF:  RMSE=0.030334 | Dir=51.9%
  LSTM base:   RMSE=0.030967 | Dir=51.9%
  Mejora ISF:  +2.04% en RMSE

SIGUIENTE: Notebook 06 — Dashboard Streamlit + Conclusiones del TFM


---
## Sección 10: Interpretación para el TFM

### ¿Qué significan los resultados del LSTM para la tesis?

**Si LSTM + ISF tiene menor RMSE que el baseline:**

> *'El modelo LSTM que incorpora el Índice de Sentimiento Financiero (ISF)
> como variable exógena logra una reducción del X% en RMSE respecto al modelo
> baseline que solo utiliza precios históricos. Este resultado confirma la hipótesis
> H_LSTM: el análisis de sentimiento basado en FinBERT aporta señales predictivas
> que los modelos lineales (Granger, Pearson) no detectan, consistente con la
> naturaleza no lineal de los mercados financieros (Lo, 2004).'*

**Sobre la métrica de dirección (Direction Accuracy):**

La dirección es más útil que RMSE en trading:
- RMSE mide el error numérico (cuánto me equivoqué en valor)
- Dirección mide si predije correctamente SUBE o BAJA
- En trading, acertar la dirección el 55-60% del tiempo es suficiente para ser rentable

**Conexión con el TFM completo:**

| Notebook | Contribución al TFM |
| --- | --- |
| NB01 | Datos de mercado reales (BTC, ETH, SPY 2021-2024) |
| NB02 | Pipeline NLP + lexicón financiero en español |
| NB03 | Comparativa de 5 modelos de sentimiento (ganador: SVM 80.9%) |
| NB04 | ISF diario + tests estadísticos (Granger, ADF, Pearson/Spearman) |
| **NB05** | **LSTM: demuestra que el sentimiento mejora la predicción de precios** |
| NB06 | Dashboard interactivo + consolidación de resultados |
